In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Optional Butterworth
try:
    from scipy.signal import butter, filtfilt
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# ----------------------------
# Daten laden
# ----------------------------
def load_npz(path):
    d = np.load(path)
    return d["X_clean"], d["X_noisy"], d["y"]


# ----------------------------
# Helper: NaN-Interpolation (damit "raw" wirklich raw bleiben kann)
# -> keine bewusste Imputation, nur nötig, um Features/FFT berechnen zu können
# ----------------------------
def interp_nan_1d(x):
    x = x.astype(float)
    n = len(x)
    if not np.isnan(x).any():
        return x
    idx = np.arange(n)
    good = ~np.isnan(x)
    if good.sum() < 2:
        return np.nan_to_num(x, nan=0.0)
    return np.interp(idx, idx[good], x[good])

def interp_nan_ts(X):
    X2 = X.copy().astype(float)
    for c in range(X2.shape[1]):
        X2[:, c] = interp_nan_1d(X2[:, c])
    return X2


# ----------------------------
# Outlier Handling
# ----------------------------
def winsorize_ts(X, low_q=0.01, high_q=0.99):
    Xw = X.copy().astype(float)
    for c in range(Xw.shape[1]):
        col = Xw[:, c]
        lo = np.quantile(col, low_q)
        hi = np.quantile(col, high_q)
        Xw[:, c] = np.clip(col, lo, hi)
    return Xw

def iso_forest_mark_outliers(X_train_ts, X_test_ts, contamination=0.02, random_state=42):
    """
    Fit IF on TRAIN only using flattened time points (C dims).
    Mark detected outliers as NaN (then MV step can handle).
    """
    Ntr, T, C = X_train_ts.shape

    flat = X_train_ts.reshape(-1, C).astype(float)
    flat = flat[~np.isnan(flat).any(axis=1)]
    if flat.shape[0] < 1000:
        return X_train_ts, X_test_ts

    m = min(30000, flat.shape[0])
    idx = np.random.default_rng(random_state).choice(flat.shape[0], size=m, replace=False)
    fit_data = flat[idx]

    iso = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1
    )
    iso.fit(fit_data)

    def apply(ts):
        out = ts.copy().astype(float)
        flat2 = out.reshape(-1, C)
        valid = ~np.isnan(flat2).any(axis=1)
        pred = np.ones(flat2.shape[0], dtype=int)
        pred[valid] = iso.predict(flat2[valid])  # -1 outlier
        flat2[pred == -1, :] = np.nan
        return flat2.reshape(out.shape)

    return apply(X_train_ts), apply(X_test_ts)


# ----------------------------
# Missing Value Handling
# ----------------------------
def mean_impute_ts(X):
    Xi = X.copy().astype(float)
    for c in range(Xi.shape[1]):
        col = Xi[:, c]
        m = np.nanmean(col)
        col[np.isnan(col)] = m if not np.isnan(m) else 0.0
        Xi[:, c] = col
    return Xi

def locf_impute_ts(X):
    Xi = X.copy().astype(float)
    for c in range(Xi.shape[1]):
        col = Xi[:, c]
        last = np.nan
        for i in range(len(col)):
            if np.isnan(col[i]) and not np.isnan(last):
                col[i] = last
            elif not np.isnan(col[i]):
                last = col[i]
        m = np.nanmean(col)
        col[np.isnan(col)] = m if not np.isnan(m) else 0.0
        Xi[:, c] = col
    return Xi

def knn_impute_ts(X, k=5):
    # (T,C) -> KNNImputer
    imp = KNNImputer(n_neighbors=k, weights="distance")
    return imp.fit_transform(X.astype(float))


# ----------------------------
# Normalisierung auf ZEITREIHE (leakage-safe: fit auf Train)
# ----------------------------
def fit_time_scaler(X_train_ts, kind="standard"):
    """
    Fit scaler on flattened TRAIN timepoints (exclude NaNs).
    """
    N, T, C = X_train_ts.shape
    flat = X_train_ts.reshape(-1, C).astype(float)
    flat = flat[~np.isnan(flat).any(axis=1)]
    scaler = RobustScaler() if kind == "robust" else StandardScaler()
    scaler.fit(flat)
    return scaler

def apply_time_scaler(X_ts, scaler):
    N, T, C = X_ts.shape
    flat = X_ts.reshape(-1, C).astype(float)
    # if NaNs exist (shouldn't after MV), fill temporarily
    if np.isnan(flat).any():
        # very conservative fallback
        col_means = np.nanmean(flat, axis=0)
        inds = np.where(np.isnan(flat))
        flat[inds] = np.take(col_means, inds[1])
    flat2 = scaler.transform(flat)
    return flat2.reshape(N, T, C)


# ----------------------------
# Glättung (FFT)
# ----------------------------
def fft_smooth_1d(x, keep_ratio=0.08):
    n = len(x)
    Xf = np.fft.rfft(x)
    k = max(2, int(len(Xf) * keep_ratio))
    Xf[k:] = 0
    return np.fft.irfft(Xf, n=n)

def fft_smooth_ts(X, keep_ratio=0.08):
    Xs = X.copy().astype(float)
    for c in range(Xs.shape[1]):
        Xs[:, c] = fft_smooth_1d(Xs[:, c], keep_ratio=keep_ratio)
    return Xs


# ----------------------------
# Filter (Butterworth / fallback moving average)
# ----------------------------
def butter_lowpass_ts(X, fs=120, cutoff_hz=8.0, order=4):
    Xs = X.copy().astype(float)
    if not HAS_SCIPY:
        win = 7
        kernel = np.ones(win)/win
        for c in range(Xs.shape[1]):
            Xs[:, c] = np.convolve(Xs[:, c], kernel, mode="same")
        return Xs

    b, a = butter(order, cutoff_hz/(fs/2), btype="low")
    for c in range(Xs.shape[1]):
        Xs[:, c] = filtfilt(b, a, Xs[:, c])
    return Xs


# ----------------------------
# Feature Extraction (Window-based)
# ----------------------------
def extract_features(X_ts, fs=120, win_sec=1.0, step_sec=0.5, fft_feats=True):
    N, T, C = X_ts.shape
    win = int(win_sec * fs)
    step = int(step_sec * fs)

    feats = []
    for i in range(N):
        x = X_ts[i]
        rows = []
        for s in range(0, T - win + 1, step):
            seg = x[s:s+win, :]

            mean = seg.mean(axis=0)
            std  = seg.std(axis=0)
            mn   = seg.min(axis=0)
            mx   = seg.max(axis=0)

            dx = np.diff(seg[:, 0], prepend=seg[0, 0])
            dy = np.diff(seg[:, 1], prepend=seg[0, 1])
            vel = np.sqrt(dx*dx + dy*dy)
            v_mean = vel.mean()
            v_std  = vel.std()

            f = np.concatenate([mean, std, mn, mx, [v_mean, v_std]])

            if fft_feats:
                for c in range(C):
                    sp = np.fft.rfft(seg[:, c])
                    mag = np.abs(sp)
                    dom = mag[1:].max() if mag.size > 1 else 0.0
                    f = np.concatenate([f, [dom]])

            rows.append(f)

        rows = np.vstack(rows)
        feats.append(np.concatenate([rows.mean(axis=0), rows.std(axis=0)]))

    return np.vstack(feats)


# ----------------------------
# RF config (fix)
# ----------------------------
def make_rf():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )


# ----------------------------
# Pipelines (wie vorher), aber Normalisierung ist time-series-scaler
# ----------------------------
PIPELINES = {
    "A_baseline": dict(outlier="none",   mv="mean", norm="standard", smooth=False, butter=False, feat_limit=None),
    "B_robust":   dict(outlier="iforest",mv="knn",  norm="robust",   smooth=False, butter=False, feat_limit=None),
    "C_signal":   dict(outlier="iforest",mv="knn",  norm="robust",   smooth=True,  butter=True,  feat_limit=None),
    "D_limit":    dict(outlier="iforest",mv="knn",  norm="robust",   smooth=True,  butter=True,  feat_limit=30),
    "E_clinical": dict(outlier="winsor", mv="locf", norm="standard", smooth=False, butter=False, feat_limit=None),
}


# ----------------------------
# Step-wise progressive evaluation (KUMULATIV, fixe Reihenfolge)
# ----------------------------
ORDER = ["raw", "outlier", "mv", "norm", "smooth", "filter", "feat_limit"]

def eval_progressive(X_noisy, y, cfg, fs=120):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rec = []

    # welche Schritte sind in dieser Pipeline wirklich aktiv?
    active = ["raw"]
    if cfg["outlier"] != "none": active.append("outlier")
    active.append("mv")
    active.append("norm")
    if cfg["smooth"]: active.append("smooth")
    if cfg["butter"]: active.append("filter")
    if cfg["feat_limit"] is not None: active.append("feat_limit")

    for fold, (tr, te) in enumerate(skf.split(X_noisy, y), start=1):
        Xtr, Xte = X_noisy[tr].astype(float), X_noisy[te].astype(float)
        ytr, yte = y[tr], y[te]

        # Start: wirklich RAW (nur Interpolation, damit Features berechenbar sind)
        cur_tr = np.array([interp_nan_ts(Xtr[i]) for i in range(Xtr.shape[0])])
        cur_te = np.array([interp_nan_ts(Xte[i]) for i in range(Xte.shape[0])])

        def fit_score(stage_name, cur_tr_ts, cur_te_ts, do_feat_limit=False):
            Xtr_feat = extract_features(cur_tr_ts, fs=fs, fft_feats=True)
            Xte_feat = extract_features(cur_te_ts, fs=fs, fft_feats=True)

            if do_feat_limit and cfg["feat_limit"] is not None:
                rf_tmp = make_rf()
                rf_tmp.fit(Xtr_feat, ytr)
                imp = rf_tmp.feature_importances_
                k = int(cfg["feat_limit"])
                idx = np.argsort(imp)[::-1][:k]
                Xtr_feat = Xtr_feat[:, idx]
                Xte_feat = Xte_feat[:, idx]

            clf = make_rf()
            clf.fit(Xtr_feat, ytr)
            prob = clf.predict_proba(Xte_feat)[:, 1]
            pred = (prob >= 0.5).astype(int)

            rec.append({
                "pipeline": pipe_name,
                "fold": fold,
                "step": stage_name,
                "step_index": ORDER.index(stage_name),
                "accuracy": accuracy_score(yte, pred),
                "f1": f1_score(yte, pred),
                "auroc": roc_auc_score(yte, prob) if len(np.unique(yte)) == 2 else np.nan
            })

        # 0) raw
        fit_score("raw", cur_tr, cur_te, do_feat_limit=False)

        # 1) outlier
        if "outlier" in active:
            if cfg["outlier"] == "iforest":
                # hier dürfen wieder NaNs entstehen -> MV Schritt kümmert sich
                cur_tr, cur_te = iso_forest_mark_outliers(cur_tr, cur_te, contamination=0.02)
            elif cfg["outlier"] == "winsor":
                cur_tr = np.array([winsorize_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
                cur_te = np.array([winsorize_ts(cur_te[i]) for i in range(cur_te.shape[0])])

            # nach Outlier: für Feature-Berechnung wieder interp (nicht MV!)
            tmp_tr = np.array([interp_nan_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
            tmp_te = np.array([interp_nan_ts(cur_te[i]) for i in range(cur_te.shape[0])])
            fit_score("outlier", tmp_tr, tmp_te, do_feat_limit=False)

        # 2) mv (jetzt echte Imputation)
        if cfg["mv"] == "mean":
            cur_tr = np.array([mean_impute_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
            cur_te = np.array([mean_impute_ts(cur_te[i]) for i in range(cur_te.shape[0])])
        elif cfg["mv"] == "locf":
            cur_tr = np.array([locf_impute_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
            cur_te = np.array([locf_impute_ts(cur_te[i]) for i in range(cur_te.shape[0])])
        elif cfg["mv"] == "knn":
            cur_tr = np.array([knn_impute_ts(cur_tr[i], k=5) for i in range(cur_tr.shape[0])])
            cur_te = np.array([knn_impute_ts(cur_te[i], k=5) for i in range(cur_te.shape[0])])

        fit_score("mv", cur_tr, cur_te, do_feat_limit=False)

        # 3) norm (fit scaler on TRAIN only)
        scaler = fit_time_scaler(cur_tr, kind=cfg["norm"])
        cur_tr = apply_time_scaler(cur_tr, scaler)
        cur_te = apply_time_scaler(cur_te, scaler)

        fit_score("norm", cur_tr, cur_te, do_feat_limit=False)

        # 4) smooth
        if cfg["smooth"]:
            cur_tr = np.array([fft_smooth_ts(cur_tr[i], keep_ratio=0.08) for i in range(cur_tr.shape[0])])
            cur_te = np.array([fft_smooth_ts(cur_te[i], keep_ratio=0.08) for i in range(cur_te.shape[0])])
            fit_score("smooth", cur_tr, cur_te, do_feat_limit=False)

        # 5) filter
        if cfg["butter"]:
            cur_tr = np.array([butter_lowpass_ts(cur_tr[i], fs=fs, cutoff_hz=8.0, order=4) for i in range(cur_tr.shape[0])])
            cur_te = np.array([butter_lowpass_ts(cur_te[i], fs=fs, cutoff_hz=8.0, order=4) for i in range(cur_te.shape[0])])
            fit_score("filter", cur_tr, cur_te, do_feat_limit=False)

        # 6) feature limit
        if cfg["feat_limit"] is not None:
            fit_score("feat_limit", cur_tr, cur_te, do_feat_limit=True)

    df = pd.DataFrame(rec)
    # nur die Stufen behalten, die in dieser Pipeline aktiv sind (chronologisch)
    df = df[df["step"].isin(active)].copy()
    df["step"] = pd.Categorical(df["step"], categories=active, ordered=True)
    return df


# ----------------------------
# Runner: alle Datensätze, Export Excel
# ----------------------------
DATASETS = {
    "medical":   "medical_timeseries.npz",
    "education": "education_timeseries.npz",
    "usability": "usability_timeseries.npz",
}

def run_all():
    all_df = []
    for ds_name, path in DATASETS.items():
        _, X_noisy, y = load_npz(path)
        for pipe_name, cfg in PIPELINES.items():
            globals()["pipe_name"] = pipe_name
            df = eval_progressive(X_noisy, y, cfg, fs=120)
            df["dataset"] = ds_name
            all_df.append(df)
    return pd.concat(all_df, ignore_index=True)

df_all = run_all()

# Wide tables (mean over folds)
acc_wide = df_all.groupby(["dataset","pipeline","step"])["accuracy"].mean().unstack("step").round(3)
f1_wide  = df_all.groupby(["dataset","pipeline","step"])["f1"].mean().unstack("step").round(3)
auc_wide = df_all.groupby(["dataset","pipeline","step"])["auroc"].mean().unstack("step").round(3)

# Deltas: Wirkung Schritt i vs i-1 (super für Diskussion!)
def make_deltas(wide):
    # compute column-wise diff in the existing step order per row
    return wide.diff(axis=1).round(3)

acc_delta = make_deltas(acc_wide)
f1_delta  = make_deltas(f1_wide)
auc_delta = make_deltas(auc_wide)

# Export
out_xlsx = "pipeline_stepwise_progressive.xlsx"
with pd.ExcelWriter(out_xlsx) as writer:
    acc_wide.to_excel(writer, sheet_name="Accuracy_Wide")
    acc_delta.to_excel(writer, sheet_name="Accuracy_Delta")
    f1_wide.to_excel(writer, sheet_name="F1_Wide")
    f1_delta.to_excel(writer, sheet_name="F1_Delta")
    auc_wide.to_excel(writer, sheet_name="AUROC_Wide")
    auc_delta.to_excel(writer, sheet_name="AUROC_Delta")
    df_all.to_excel(writer, sheet_name="Long_AllFolds", index=False)

print("Fertig. Export:", out_xlsx)

# optional anzeigen
display(acc_wide)
display(acc_delta)


Fertig. Export: pipeline_stepwise_progressive.xlsx


step                  feat_limit  filter    mv  norm  outlier   raw  smooth
dataset   pipeline                                                         
education A_baseline         NaN     NaN  0.68  0.74      NaN  0.68     NaN
          B_robust           NaN     NaN  0.54  0.52     0.58  0.68     NaN
          C_signal           NaN    0.48  0.54  0.52     0.58  0.68    0.48
          D_limit           0.44    0.48  0.54  0.52     0.58  0.68    0.48
          E_clinical         NaN     NaN  0.62  0.64     0.62  0.68     NaN
medical   A_baseline         NaN     NaN  0.44  0.48      NaN  0.44     NaN
          B_robust           NaN     NaN  0.48  0.52     0.44  0.44     NaN
          C_signal           NaN    0.56  0.48  0.52     0.44  0.44    0.56
          D_limit           0.56    0.56  0.48  0.52     0.44  0.44    0.56
          E_clinical         NaN     NaN  0.46  0.48     0.46  0.44     NaN
usability A_baseline         NaN     NaN  0.40  0.38      NaN  0.40     NaN
          B_robust           NaN     NaN  0.32  0.34     0.44  0.40     NaN
          C_signal           NaN    0.38  0.32  0.34     0.44  0.40    0.38
          D_limit           0.36    0.38  0.32  0.34     0.44  0.40    0.38
          E_clinical         NaN     NaN  0.40  0.44     0.40  0.40     NaN

step                  feat_limit  filter    mv  norm  outlier   raw  smooth
dataset   pipeline                                                         
education A_baseline         NaN     NaN   NaN  0.06      NaN   NaN     NaN
          B_robust           NaN     NaN   NaN -0.02     0.06  0.10     NaN
          C_signal           NaN     NaN  0.06 -0.02     0.06  0.10   -0.20
          D_limit            NaN    0.04  0.06 -0.02     0.06  0.10   -0.20
          E_clinical         NaN     NaN   NaN  0.02    -0.02  0.06     NaN
medical   A_baseline         NaN     NaN   NaN  0.04      NaN   NaN     NaN
          B_robust           NaN     NaN   NaN  0.04    -0.08  0.00     NaN
          C_signal           NaN     NaN -0.08  0.04    -0.08  0.00    0.12
          D_limit            NaN    0.00 -0.08  0.04    -0.08  0.00    0.12
          E_clinical         NaN     NaN   NaN  0.02    -0.02 -0.02     NaN
usability A_baseline         NaN     NaN   NaN -0.02      NaN   NaN     NaN
          B_robust           NaN     NaN   NaN  0.02     0.10 -0.04     NaN
          C_signal           NaN     NaN -0.06  0.02     0.10 -0.04   -0.02
          D_limit            NaN    0.02 -0.06  0.02     0.10 -0.04   -0.02
          E_clinical         NaN     NaN   NaN  0.04    -0.04  0.00     NaN

In [2]:
# ============================================================
# FULL NOTEBOOK CODE
# Step-wise, progressive (cumulative) preprocessing benchmark
# 3 datasets × (A–O pipelines) × step-wise Accuracy/F1/AUROC
# Exports: pipeline_stepwise_progressive_AO.xlsx
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Optional: Butterworth, median filter (scipy)
try:
    from scipy.signal import butter, filtfilt, medfilt
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# ----------------------------
# 0) Load data
# ----------------------------
def load_npz(path):
    d = np.load(path)
    return d["X_clean"], d["X_noisy"], d["y"]


# ----------------------------
# 1) Utility: minimal interpolation (ONLY to make features computable)
#     This is NOT the "mv step". It's a minimal baseline for "raw/outlier" scoring.
# ----------------------------
def interp_nan_1d(x):
    x = x.astype(float)
    n = len(x)
    if not np.isnan(x).any():
        return x
    idx = np.arange(n)
    good = ~np.isnan(x)
    if good.sum() < 2:
        return np.nan_to_num(x, nan=0.0)
    return np.interp(idx, idx[good], x[good])

def interp_nan_ts(X):
    # X: (T,C)
    X2 = X.copy().astype(float)
    for c in range(X2.shape[1]):
        X2[:, c] = interp_nan_1d(X2[:, c])
    return X2


# ----------------------------
# 2) Outlier handling
# ----------------------------
def winsorize_ts(X, low_q=0.01, high_q=0.99):
    Xw = X.copy().astype(float)
    for c in range(Xw.shape[1]):
        col = Xw[:, c]
        valid = col[~np.isnan(col)]
        if valid.size < 10:
            continue
        lo = np.quantile(valid, low_q)
        hi = np.quantile(valid, high_q)
        col = np.clip(col, lo, hi)
        Xw[:, c] = col
    return Xw

def iqr_clip_ts(X, k=1.5):
    """IQR-based clipping per channel (mild)."""
    Xc = X.copy().astype(float)
    for c in range(Xc.shape[1]):
        col = Xc[:, c]
        valid = col[~np.isnan(col)]
        if valid.size < 10:
            continue
        q1 = np.quantile(valid, 0.25)
        q3 = np.quantile(valid, 0.75)
        iqr = q3 - q1
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        Xc[:, c] = np.clip(col, lo, hi)
    return Xc

def iso_forest_mark_outliers(X_train_ts, X_test_ts, contamination=0.02, random_state=42):
    """
    Fit IsolationForest on TRAIN only using flattened valid timepoints.
    Mark predicted outliers in TRAIN+TEST as NaN (MV step will handle).
    """
    Ntr, T, C = X_train_ts.shape

    flat = X_train_ts.reshape(-1, C).astype(float)
    flat = flat[~np.isnan(flat).any(axis=1)]
    if flat.shape[0] < 1000:
        return X_train_ts, X_test_ts

    m = min(30000, flat.shape[0])
    idx = np.random.default_rng(random_state).choice(flat.shape[0], size=m, replace=False)
    fit_data = flat[idx]

    iso = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=random_state,
        n_jobs=-1
    )
    iso.fit(fit_data)

    def apply(ts):
        out = ts.copy().astype(float)
        flat2 = out.reshape(-1, C)
        valid = ~np.isnan(flat2).any(axis=1)
        pred = np.ones(flat2.shape[0], dtype=int)
        pred[valid] = iso.predict(flat2[valid])  # -1 outlier
        flat2[pred == -1, :] = np.nan
        return flat2.reshape(out.shape)

    return apply(X_train_ts), apply(X_test_ts)


# ----------------------------
# 3) Missing value handling (the real "mv step")
# ----------------------------
def mean_impute_ts(X):
    Xi = X.copy().astype(float)
    for c in range(Xi.shape[1]):
        col = Xi[:, c]
        m = np.nanmean(col)
        col[np.isnan(col)] = m if not np.isnan(m) else 0.0
        Xi[:, c] = col
    return Xi

def locf_impute_ts(X):
    Xi = X.copy().astype(float)
    for c in range(Xi.shape[1]):
        col = Xi[:, c]
        last = np.nan
        for i in range(len(col)):
            if np.isnan(col[i]) and not np.isnan(last):
                col[i] = last
            elif not np.isnan(col[i]):
                last = col[i]
        m = np.nanmean(col)
        col[np.isnan(col)] = m if not np.isnan(m) else 0.0
        Xi[:, c] = col
    return Xi

def knn_impute_ts(X, k=5):
    imp = KNNImputer(n_neighbors=k, weights="distance")
    return imp.fit_transform(X.astype(float))

def linear_interp_impute_ts(X):
    Xi = X.copy().astype(float)
    T, C = Xi.shape
    idx = np.arange(T)
    for c in range(C):
        col = Xi[:, c]
        good = ~np.isnan(col)
        if good.sum() < 2:
            m = np.nanmean(col)
            col = np.nan_to_num(col, nan=m if not np.isnan(m) else 0.0)
        else:
            col = np.interp(idx, idx[good], col[good])
        Xi[:, c] = col
    return Xi


# ----------------------------
# 4) Normalization on TIME SERIES (leakage-safe)
# ----------------------------
def fit_time_scaler(X_train_ts, kind="standard"):
    N, T, C = X_train_ts.shape
    flat = X_train_ts.reshape(-1, C).astype(float)
    flat = flat[~np.isnan(flat).any(axis=1)]
    scaler = RobustScaler() if kind == "robust" else StandardScaler()
    scaler.fit(flat)
    return scaler

def apply_time_scaler(X_ts, scaler):
    N, T, C = X_ts.shape
    flat = X_ts.reshape(-1, C).astype(float)
    if np.isnan(flat).any():
        col_means = np.nanmean(flat, axis=0)
        inds = np.where(np.isnan(flat))
        flat[inds] = np.take(col_means, inds[1])
    flat2 = scaler.transform(flat)
    return flat2.reshape(N, T, C)


# ----------------------------
# 5) Smoothing
# ----------------------------
def fft_smooth_1d(x, keep_ratio=0.08):
    n = len(x)
    Xf = np.fft.rfft(x)
    k = max(2, int(len(Xf) * keep_ratio))
    Xf[k:] = 0
    return np.fft.irfft(Xf, n=n)

def fft_smooth_ts(X, keep_ratio=0.08):
    Xs = X.copy().astype(float)
    for c in range(Xs.shape[1]):
        Xs[:, c] = fft_smooth_1d(Xs[:, c], keep_ratio=keep_ratio)
    return Xs

def median_filter_ts(X, k=5):
    Xf = X.copy().astype(float)
    if HAS_SCIPY:
        for c in range(Xf.shape[1]):
            Xf[:, c] = medfilt(Xf[:, c], kernel_size=k)
        return Xf
    # fallback: rolling median (slow, but ok for small runs)
    pad = k // 2
    for c in range(Xf.shape[1]):
        col = Xf[:, c]
        out = col.copy()
        for i in range(len(col)):
            a = max(0, i - pad)
            b = min(len(col), i + pad + 1)
            out[i] = np.median(col[a:b])
        Xf[:, c] = out
    return Xf


# ----------------------------
# 6) Filter: Butterworth low-pass (or moving average fallback)
# ----------------------------
def butter_lowpass_ts(X, fs=120, cutoff_hz=8.0, order=4):
    Xs = X.copy().astype(float)
    if not HAS_SCIPY:
        win = 7
        kernel = np.ones(win) / win
        for c in range(Xs.shape[1]):
            Xs[:, c] = np.convolve(Xs[:, c], kernel, mode="same")
        return Xs
    b, a = butter(order, cutoff_hz / (fs / 2), btype="low")
    for c in range(Xs.shape[1]):
        Xs[:, c] = filtfilt(b, a, Xs[:, c])
    return Xs


# ----------------------------
# 7) Feature extraction (window-based)
# ----------------------------
def extract_features(X_ts, fs=120, win_sec=1.0, step_sec=0.5, fft_feats=True):
    """
    X_ts: (N,T,C) with no NaNs preferred (after mv step).
    """
    N, T, C = X_ts.shape
    win = int(win_sec * fs)
    step = int(step_sec * fs)

    feats = []
    for i in range(N):
        x = X_ts[i]
        rows = []
        for s in range(0, T - win + 1, step):
            seg = x[s:s + win, :]  # (win,C)

            mean = seg.mean(axis=0)
            std  = seg.std(axis=0)
            mn   = seg.min(axis=0)
            mx   = seg.max(axis=0)

            dx = np.diff(seg[:, 0], prepend=seg[0, 0])
            dy = np.diff(seg[:, 1], prepend=seg[0, 1])
            vel = np.sqrt(dx * dx + dy * dy)
            v_mean = vel.mean()
            v_std  = vel.std()

            f = np.concatenate([mean, std, mn, mx, [v_mean, v_std]])

            if fft_feats:
                for c in range(C):
                    sp = np.fft.rfft(seg[:, c])
                    mag = np.abs(sp)
                    dom = mag[1:].max() if mag.size > 1 else 0.0
                    f = np.concatenate([f, [dom]])

            rows.append(f)

        rows = np.vstack(rows)
        feats.append(np.concatenate([rows.mean(axis=0), rows.std(axis=0)]))

    return np.vstack(feats)


# ----------------------------
# 8) Random Forest (fixed config)
# ----------------------------
def make_rf():
    return RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        bootstrap=True,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )


# ----------------------------
# 9) Pipelines A–O
#   Fixed step order: raw → outlier → mv → norm → smooth → filter → feat_limit
#   smooth: "none" | "fft" | "median"
# ----------------------------
PIPELINES = {
    # Original 5
    "A_baseline":  dict(outlier="none",     mv="mean",          norm="standard", smooth="none",   butter=False, feat_limit=None),
    "B_robust":    dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="none",   butter=False, feat_limit=None),
    "C_signal":    dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="fft",    butter=True,  feat_limit=None),
    "D_limit":     dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="fft",    butter=True,  feat_limit=30),
    "E_clinical":  dict(outlier="winsor",   mv="locf",          norm="standard", smooth="none",   butter=False, feat_limit=None),

    # +10 new
    "F_knn_only":      dict(outlier="none",     mv="knn",           norm="robust",   smooth="none",   butter=False, feat_limit=None),
    "G_iqr_knn":       dict(outlier="iqr_clip", mv="knn",           norm="robust",   smooth="none",   butter=False, feat_limit=None),
    "H_if_mean":       dict(outlier="iforest",  mv="mean",          norm="robust",   smooth="none",   butter=False, feat_limit=None),
    "I_locf_robust":   dict(outlier="none",     mv="locf",          norm="robust",   smooth="none",   butter=False, feat_limit=None),
    "J_interp_std":    dict(outlier="none",     mv="linear_interp", norm="standard", smooth="none",   butter=False, feat_limit=None),
    "K_norm_only":     dict(outlier="none",     mv="mean",          norm="robust",   smooth="none",   butter=False, feat_limit=None),
    "L_fft_only":      dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="fft",    butter=False, feat_limit=None),
    "M_butter_only":   dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="none",   butter=True,  feat_limit=None),
    "N_median_smooth": dict(outlier="none",     mv="linear_interp", norm="robust",   smooth="median", butter=False, feat_limit=None),
    "O_limit10":       dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="fft",    butter=True,  feat_limit=10),
}


# ----------------------------
# 10) Progressive evaluation (cumulative, fixed order)
# ----------------------------
ORDER = ["raw", "outlier", "mv", "norm", "smooth", "filter", "feat_limit"]

def eval_progressive(X_noisy, y, cfg, fs=120):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rec = []

    # active steps for this pipeline (always include raw, mv, norm; optional others)
    active = ["raw"]
    if cfg["outlier"] != "none":
        active.append("outlier")
    active.append("mv")
    active.append("norm")
    if cfg["smooth"] != "none":
        active.append("smooth")
    if cfg["butter"]:
        active.append("filter")
    if cfg["feat_limit"] is not None:
        active.append("feat_limit")

    for fold, (tr, te) in enumerate(skf.split(X_noisy, y), start=1):
        Xtr = X_noisy[tr].astype(float)
        Xte = X_noisy[te].astype(float)
        ytr = y[tr]
        yte = y[te]

        # RAW baseline: only minimal interpolation so features compute
        cur_tr = np.array([interp_nan_ts(Xtr[i]) for i in range(Xtr.shape[0])])
        cur_te = np.array([interp_nan_ts(Xte[i]) for i in range(Xte.shape[0])])

        def fit_score(stage_name, cur_tr_ts, cur_te_ts, do_feat_limit=False):
            Xtr_feat = extract_features(cur_tr_ts, fs=fs, fft_feats=True)
            Xte_feat = extract_features(cur_te_ts, fs=fs, fft_feats=True)

            if do_feat_limit and cfg["feat_limit"] is not None:
                rf_tmp = make_rf()
                rf_tmp.fit(Xtr_feat, ytr)
                imp = rf_tmp.feature_importances_
                k = int(cfg["feat_limit"])
                idx = np.argsort(imp)[::-1][:k]
                Xtr_feat = Xtr_feat[:, idx]
                Xte_feat = Xte_feat[:, idx]

            clf = make_rf()
            clf.fit(Xtr_feat, ytr)
            prob = clf.predict_proba(Xte_feat)[:, 1]
            pred = (prob >= 0.5).astype(int)

            try:
                auc = roc_auc_score(yte, prob)
            except Exception:
                auc = np.nan

            rec.append({
                "pipeline": pipe_name,
                "fold": fold,
                "step": stage_name,
                "step_index": ORDER.index(stage_name),
                "accuracy": accuracy_score(yte, pred),
                "f1": f1_score(yte, pred),
                "auroc": auc
            })

        # --- 0) raw
        fit_score("raw", cur_tr, cur_te, do_feat_limit=False)

        # --- 1) outlier (may reintroduce NaNs if iforest)
        if cfg["outlier"] != "none":
            if cfg["outlier"] == "iforest":
                cur_tr, cur_te = iso_forest_mark_outliers(cur_tr, cur_te, contamination=0.02)
            elif cfg["outlier"] == "winsor":
                cur_tr = np.array([winsorize_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
                cur_te = np.array([winsorize_ts(cur_te[i]) for i in range(cur_te.shape[0])])
            elif cfg["outlier"] == "iqr_clip":
                cur_tr = np.array([iqr_clip_ts(cur_tr[i], k=1.5) for i in range(cur_tr.shape[0])])
                cur_te = np.array([iqr_clip_ts(cur_te[i], k=1.5) for i in range(cur_te.shape[0])])
            else:
                raise ValueError(f"Unknown outlier mode: {cfg['outlier']}")

            # score after outlier: still only minimal interpolation for feature computation
            tmp_tr = np.array([interp_nan_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
            tmp_te = np.array([interp_nan_ts(cur_te[i]) for i in range(cur_te.shape[0])])
            fit_score("outlier", tmp_tr, tmp_te, do_feat_limit=False)

        # --- 2) mv (real imputation)
        if cfg["mv"] == "mean":
            cur_tr = np.array([mean_impute_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
            cur_te = np.array([mean_impute_ts(cur_te[i]) for i in range(cur_te.shape[0])])
        elif cfg["mv"] == "locf":
            cur_tr = np.array([locf_impute_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
            cur_te = np.array([locf_impute_ts(cur_te[i]) for i in range(cur_te.shape[0])])
        elif cfg["mv"] == "knn":
            cur_tr = np.array([knn_impute_ts(cur_tr[i], k=5) for i in range(cur_tr.shape[0])])
            cur_te = np.array([knn_impute_ts(cur_te[i], k=5) for i in range(cur_te.shape[0])])
        elif cfg["mv"] == "linear_interp":
            cur_tr = np.array([linear_interp_impute_ts(cur_tr[i]) for i in range(cur_tr.shape[0])])
            cur_te = np.array([linear_interp_impute_ts(cur_te[i]) for i in range(cur_te.shape[0])])
        else:
            raise ValueError(f"Unknown mv mode: {cfg['mv']}")

        fit_score("mv", cur_tr, cur_te, do_feat_limit=False)

        # --- 3) norm (leakage-safe, fitted on train only)
        scaler = fit_time_scaler(cur_tr, kind=cfg["norm"])
        cur_tr = apply_time_scaler(cur_tr, scaler)
        cur_te = apply_time_scaler(cur_te, scaler)

        fit_score("norm", cur_tr, cur_te, do_feat_limit=False)

        # --- 4) smooth
        if cfg["smooth"] != "none":
            if cfg["smooth"] == "fft":
                cur_tr = np.array([fft_smooth_ts(cur_tr[i], keep_ratio=0.08) for i in range(cur_tr.shape[0])])
                cur_te = np.array([fft_smooth_ts(cur_te[i], keep_ratio=0.08) for i in range(cur_te.shape[0])])
            elif cfg["smooth"] == "median":
                cur_tr = np.array([median_filter_ts(cur_tr[i], k=5) for i in range(cur_tr.shape[0])])
                cur_te = np.array([median_filter_ts(cur_te[i], k=5) for i in range(cur_te.shape[0])])
            else:
                raise ValueError(f"Unknown smooth mode: {cfg['smooth']}")
            fit_score("smooth", cur_tr, cur_te, do_feat_limit=False)

        # --- 5) filter
        if cfg["butter"]:
            cur_tr = np.array([butter_lowpass_ts(cur_tr[i], fs=fs, cutoff_hz=8.0, order=4) for i in range(cur_tr.shape[0])])
            cur_te = np.array([butter_lowpass_ts(cur_te[i], fs=fs, cutoff_hz=8.0, order=4) for i in range(cur_te.shape[0])])
            fit_score("filter", cur_tr, cur_te, do_feat_limit=False)

        # --- 6) feature limit
        if cfg["feat_limit"] is not None:
            fit_score("feat_limit", cur_tr, cur_te, do_feat_limit=True)

    df = pd.DataFrame(rec)
    df = df[df["step"].isin(active)].copy()
    df["step"] = pd.Categorical(df["step"], categories=active, ordered=True)
    return df


# ----------------------------
# 11) Run everything + export Excel
# ----------------------------
DATASETS = {
    "medical":   "medical_timeseries.npz",
    "education": "education_timeseries.npz",
    "usability": "usability_timeseries.npz",
}

def run_all():
    all_df = []
    for ds_name, path in DATASETS.items():
        _, X_noisy, y = load_npz(path)
        for pipe_name, cfg in PIPELINES.items():
            globals()["pipe_name"] = pipe_name
            df = eval_progressive(X_noisy, y, cfg, fs=120)
            df["dataset"] = ds_name
            all_df.append(df)
            print(f"Done: {ds_name} / {pipe_name}")
    return pd.concat(all_df, ignore_index=True)

df_all = run_all()

# Wide means over folds
acc_wide = df_all.groupby(["dataset","pipeline","step"])["accuracy"].mean().unstack("step").round(3)
f1_wide  = df_all.groupby(["dataset","pipeline","step"])["f1"].mean().unstack("step").round(3)
auc_wide = df_all.groupby(["dataset","pipeline","step"])["auroc"].mean().unstack("step").round(3)

# Deltas (marginal gain per added step)
def make_deltas(wide):
    return wide.diff(axis=1).round(3)

acc_delta = make_deltas(acc_wide)
f1_delta  = make_deltas(f1_wide)
auc_delta = make_deltas(auc_wide)

# Export
out_xlsx = "pipeline_stepwise_progressive_AO.xlsx"
with pd.ExcelWriter(out_xlsx) as writer:
    acc_wide.to_excel(writer, sheet_name="Accuracy_Wide")
    acc_delta.to_excel(writer, sheet_name="Accuracy_Delta")
    f1_wide.to_excel(writer, sheet_name="F1_Wide")
    f1_delta.to_excel(writer, sheet_name="F1_Delta")
    auc_wide.to_excel(writer, sheet_name="AUROC_Wide")
    auc_delta.to_excel(writer, sheet_name="AUROC_Delta")
    df_all.to_excel(writer, sheet_name="Long_AllFolds", index=False)

print("\nEXPORT:", out_xlsx)

# Display a quick view
display(acc_wide.head(20))
display(acc_delta.head(20))


Done: medical / A_baseline
Done: medical / B_robust
Done: medical / C_signal
Done: medical / D_limit
Done: medical / E_clinical
Done: medical / F_knn_only
Done: medical / G_iqr_knn
Done: medical / H_if_mean
Done: medical / I_locf_robust
Done: medical / J_interp_std
Done: medical / K_norm_only
Done: medical / L_fft_only
Done: medical / M_butter_only
Done: medical / N_median_smooth
Done: medical / O_limit10
Done: education / A_baseline
Done: education / B_robust
Done: education / C_signal
Done: education / D_limit
Done: education / E_clinical
Done: education / F_knn_only
Done: education / G_iqr_knn
Done: education / H_if_mean
Done: education / I_locf_robust
Done: education / J_interp_std
Done: education / K_norm_only
Done: education / L_fft_only
Done: education / M_butter_only
Done: education / N_median_smooth
Done: education / O_limit10
Done: usability / A_baseline
Done: usability / B_robust
Done: usability / C_signal
Done: usability / D_limit
Done: usability / E_clinical
Done: usabilit

step                       feat_limit  filter    mv  norm  outlier   raw  \
dataset   pipeline                                                         
education A_baseline              NaN     NaN  0.68  0.74      NaN  0.68   
          B_robust                NaN     NaN  0.54  0.52     0.58  0.68   
          C_signal                NaN    0.48  0.54  0.52     0.58  0.68   
          D_limit                0.44    0.48  0.54  0.52     0.58  0.68   
          E_clinical              NaN     NaN  0.62  0.64     0.62  0.68   
          F_knn_only              NaN     NaN  0.68  0.70      NaN  0.68   
          G_iqr_knn               NaN     NaN  0.76  0.76     0.76  0.68   
          H_if_mean               NaN     NaN  0.54  0.52     0.58  0.68   
          I_locf_robust           NaN     NaN  0.68  0.70      NaN  0.68   
          J_interp_std            NaN     NaN  0.68  0.74      NaN  0.68   
          K_norm_only             NaN     NaN  0.68  0.70      NaN  0.68   
          L_fft_only              NaN     NaN  0.54  0.52     0.58  0.68   
          M_butter_only           NaN    0.50  0.54  0.52     0.58  0.68   
          N_median_smooth         NaN     NaN  0.68  0.70      NaN  0.68   
          O_limit10              0.50    0.48  0.54  0.52     0.58  0.68   
medical   A_baseline              NaN     NaN  0.44  0.48      NaN  0.44   
          B_robust                NaN     NaN  0.48  0.52     0.44  0.44   
          C_signal                NaN    0.56  0.48  0.52     0.44  0.44   
          D_limit                0.56    0.56  0.48  0.52     0.44  0.44   
          E_clinical              NaN     NaN  0.46  0.48     0.46  0.44   

step                       smooth  
dataset   pipeline                 
education A_baseline          NaN  
          B_robust            NaN  
          C_signal           0.48  
          D_limit            0.48  
          E_clinical          NaN  
          F_knn_only          NaN  
          G_iqr_knn           NaN  
          H_if_mean           NaN  
          I_locf_robust       NaN  
          J_interp_std        NaN  
          K_norm_only         NaN  
          L_fft_only         0.48  
          M_butter_only       NaN  
          N_median_smooth    0.60  
          O_limit10          0.48  
medical   A_baseline          NaN  
          B_robust            NaN  
          C_signal           0.56  
          D_limit            0.56  
          E_clinical          NaN

step                       feat_limit  filter    mv  norm  outlier   raw  \
dataset   pipeline                                                         
education A_baseline              NaN     NaN   NaN  0.06      NaN   NaN   
          B_robust                NaN     NaN   NaN -0.02     0.06  0.10   
          C_signal                NaN     NaN  0.06 -0.02     0.06  0.10   
          D_limit                 NaN    0.04  0.06 -0.02     0.06  0.10   
          E_clinical              NaN     NaN   NaN  0.02    -0.02  0.06   
          F_knn_only              NaN     NaN   NaN  0.02      NaN   NaN   
          G_iqr_knn               NaN     NaN   NaN  0.00     0.00 -0.08   
          H_if_mean               NaN     NaN   NaN -0.02     0.06  0.10   
          I_locf_robust           NaN     NaN   NaN  0.02      NaN   NaN   
          J_interp_std            NaN     NaN   NaN  0.06      NaN   NaN   
          K_norm_only             NaN     NaN   NaN  0.02      NaN   NaN   
          L_fft_only              NaN     NaN   NaN -0.02     0.06  0.10   
          M_butter_only           NaN     NaN  0.04 -0.02     0.06  0.10   
          N_median_smooth         NaN     NaN   NaN  0.02      NaN   NaN   
          O_limit10               NaN   -0.02  0.06 -0.02     0.06  0.10   
medical   A_baseline              NaN     NaN   NaN  0.04      NaN   NaN   
          B_robust                NaN     NaN   NaN  0.04    -0.08  0.00   
          C_signal                NaN     NaN -0.08  0.04    -0.08  0.00   
          D_limit                 NaN    0.00 -0.08  0.04    -0.08  0.00   
          E_clinical              NaN     NaN   NaN  0.02    -0.02 -0.02   

step                       smooth  
dataset   pipeline                 
education A_baseline          NaN  
          B_robust            NaN  
          C_signal          -0.20  
          D_limit           -0.20  
          E_clinical          NaN  
          F_knn_only          NaN  
          G_iqr_knn           NaN  
          H_if_mean           NaN  
          I_locf_robust       NaN  
          J_interp_std        NaN  
          K_norm_only         NaN  
          L_fft_only        -0.20  
          M_butter_only       NaN  
          N_median_smooth   -0.08  
          O_limit10         -0.20  
medical   A_baseline          NaN  
          B_robust            NaN  
          C_signal           0.12  
          D_limit            0.12  
          E_clinical          NaN

In [2]:
# ============================================================
# FULL NOTEBOOK: MV-FIRST ORDER
# raw → mv → outlier → norm → smooth → filter → feat_limit
# Pipelines A–O
# Export: pipeline_stepwise_progressive_AO_MVfirst.xlsx
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Optional scipy
try:
    from scipy.signal import butter, filtfilt, medfilt
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False


# ============================================================
# 1) LOAD DATA
# ============================================================

def load_npz(path):
    d = np.load(path)
    return d["X_clean"], d["X_noisy"], d["y"]


# ============================================================
# 2) MINIMAL INTERPOLATION (only for RAW scoring)
# ============================================================

def interp_nan_1d(x):
    x = x.astype(float)
    n = len(x)
    if not np.isnan(x).any():
        return x
    idx = np.arange(n)
    good = ~np.isnan(x)
    if good.sum() < 2:
        return np.nan_to_num(x, nan=0.0)
    return np.interp(idx, idx[good], x[good])

def interp_nan_ts(X):
    X2 = X.copy().astype(float)
    for c in range(X2.shape[1]):
        X2[:, c] = interp_nan_1d(X2[:, c])
    return X2


# ============================================================
# 3) OUTLIER METHODS
# ============================================================

def winsorize_ts(X, low_q=0.01, high_q=0.99):
    Xw = X.copy().astype(float)
    for c in range(Xw.shape[1]):
        col = Xw[:, c]
        valid = col[~np.isnan(col)]
        if valid.size < 10:
            continue
        lo = np.quantile(valid, low_q)
        hi = np.quantile(valid, high_q)
        Xw[:, c] = np.clip(col, lo, hi)
    return Xw

def iqr_clip_ts(X, k=1.5):
    Xc = X.copy().astype(float)
    for c in range(Xc.shape[1]):
        col = Xc[:, c]
        valid = col[~np.isnan(col)]
        if valid.size < 10:
            continue
        q1 = np.quantile(valid, 0.25)
        q3 = np.quantile(valid, 0.75)
        iqr = q3 - q1
        lo = q1 - k * iqr
        hi = q3 + k * iqr
        Xc[:, c] = np.clip(col, lo, hi)
    return Xc

def iso_forest_mark_outliers(X_train_ts, X_test_ts, contamination=0.02):
    Ntr, T, C = X_train_ts.shape
    flat = X_train_ts.reshape(-1, C).astype(float)
    flat = flat[~np.isnan(flat).any(axis=1)]
    if flat.shape[0] < 1000:
        return X_train_ts, X_test_ts

    iso = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=42,
        n_jobs=-1
    )
    iso.fit(flat)

    def apply(ts):
        out = ts.copy().astype(float)
        flat2 = out.reshape(-1, C)
        valid = ~np.isnan(flat2).any(axis=1)
        pred = np.ones(flat2.shape[0])
        pred[valid] = iso.predict(flat2[valid])
        flat2[pred == -1] = np.nan
        return flat2.reshape(out.shape)

    return apply(X_train_ts), apply(X_test_ts)


# ============================================================
# 4) MISSING VALUE METHODS
# ============================================================

def mean_impute_ts(X):
    Xi = X.copy().astype(float)
    for c in range(Xi.shape[1]):
        col = Xi[:, c]
        m = np.nanmean(col)
        col[np.isnan(col)] = m if not np.isnan(m) else 0.0
        Xi[:, c] = col
    return Xi

def locf_impute_ts(X):
    Xi = X.copy().astype(float)
    for c in range(Xi.shape[1]):
        col = Xi[:, c]
        last = np.nan
        for i in range(len(col)):
            if np.isnan(col[i]) and not np.isnan(last):
                col[i] = last
            elif not np.isnan(col[i]):
                last = col[i]
        m = np.nanmean(col)
        col[np.isnan(col)] = m if not np.isnan(m) else 0.0
        Xi[:, c] = col
    return Xi

def knn_impute_ts(X, k=5):
    imp = KNNImputer(n_neighbors=k, weights="distance")
    return imp.fit_transform(X.astype(float))

def linear_interp_impute_ts(X):
    Xi = X.copy().astype(float)
    T, C = Xi.shape
    idx = np.arange(T)
    for c in range(C):
        col = Xi[:, c]
        good = ~np.isnan(col)
        if good.sum() < 2:
            m = np.nanmean(col)
            col = np.nan_to_num(col, nan=m if not np.isnan(m) else 0.0)
        else:
            col = np.interp(idx, idx[good], col[good])
        Xi[:, c] = col
    return Xi


# ============================================================
# 5) NORMALIZATION (LEAKAGE SAFE)
# ============================================================

def fit_time_scaler(X_train_ts, kind="standard"):
    N, T, C = X_train_ts.shape
    flat = X_train_ts.reshape(-1, C)
    scaler = RobustScaler() if kind == "robust" else StandardScaler()
    scaler.fit(flat)
    return scaler

def apply_time_scaler(X_ts, scaler):
    N, T, C = X_ts.shape
    flat = X_ts.reshape(-1, C)
    flat2 = scaler.transform(flat)
    return flat2.reshape(N, T, C)


# ============================================================
# 6) SMOOTHING / FILTER
# ============================================================

def fft_smooth_ts(X, keep_ratio=0.08):
    Xs = X.copy().astype(float)
    for c in range(Xs.shape[1]):
        x = Xs[:, c]
        Xf = np.fft.rfft(x)
        k = max(2, int(len(Xf) * keep_ratio))
        Xf[k:] = 0
        Xs[:, c] = np.fft.irfft(Xf, n=len(x))
    return Xs

def median_filter_ts(X, k=5):
    Xf = X.copy().astype(float)
    if HAS_SCIPY:
        for c in range(Xf.shape[1]):
            Xf[:, c] = medfilt(Xf[:, c], kernel_size=k)
    return Xf

def butter_lowpass_ts(X, fs=120, cutoff_hz=8.0):
    Xs = X.copy().astype(float)
    if not HAS_SCIPY:
        return Xs
    b, a = butter(4, cutoff_hz/(fs/2), btype="low")
    for c in range(Xs.shape[1]):
        Xs[:, c] = filtfilt(b, a, Xs[:, c])
    return Xs


# ============================================================
# 7) FEATURE EXTRACTION
# ============================================================

def extract_features(X_ts, fs=120):
    N, T, C = X_ts.shape
    win = int(1.0 * fs)
    step = int(0.5 * fs)

    feats = []
    for i in range(N):
        x = X_ts[i]
        rows = []
        for s in range(0, T - win + 1, step):
            seg = x[s:s+win]
            mean = seg.mean(axis=0)
            std  = seg.std(axis=0)
            f = np.concatenate([mean, std])
            rows.append(f)
        rows = np.vstack(rows)
        feats.append(np.concatenate([rows.mean(axis=0), rows.std(axis=0)]))
    return np.vstack(feats)


# ============================================================
# 8) MODEL
# ============================================================

def make_rf():
    return RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )


# ============================================================
# 9) PIPELINES (A–O)
# ============================================================

PIPELINES = {
    "A_baseline":  dict(outlier="none",     mv="mean",          norm="standard", smooth="none",   butter=False, feat_limit=None),
    "B_robust":    dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="none",   butter=False, feat_limit=None),
    "C_signal":    dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="fft",    butter=True,  feat_limit=None),
    "D_limit":     dict(outlier="iforest",  mv="knn",           norm="robust",   smooth="fft",    butter=True,  feat_limit=30),
    "E_clinical":  dict(outlier="winsor",   mv="locf",          norm="standard", smooth="none",   butter=False, feat_limit=None),
}


# ============================================================
# 10) MV-FIRST EVALUATION
# ============================================================

ORDER = ["raw","mv","outlier","norm","smooth","filter","feat_limit"]

def eval_mv_first(X_noisy, y, cfg):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    records = []

    for fold,(tr,te) in enumerate(skf.split(X_noisy,y),1):
        Xtr = X_noisy[tr]
        Xte = X_noisy[te]
        ytr = y[tr]
        yte = y[te]

        # RAW
        raw_tr = np.array([interp_nan_ts(Xtr[i]) for i in range(len(Xtr))])
        raw_te = np.array([interp_nan_ts(Xte[i]) for i in range(len(Xte))])

        def score(stage, cur_tr, cur_te):
            Xtr_feat = extract_features(cur_tr)
            Xte_feat = extract_features(cur_te)
            clf = make_rf()
            clf.fit(Xtr_feat,ytr)
            pred = clf.predict(Xte_feat)
            records.append(dict(
                pipeline=pipe_name,
                fold=fold,
                step=stage,
                accuracy=accuracy_score(yte,pred)
            ))

        score("raw",raw_tr,raw_te)

        # MV FIRST
        cur_tr = np.array([mean_impute_ts(Xtr[i]) for i in range(len(Xtr))])
        cur_te = np.array([mean_impute_ts(Xte[i]) for i in range(len(Xte))])
        score("mv",cur_tr,cur_te)

        # OUTLIER AFTER MV
        if cfg["outlier"]=="iforest":
            cur_tr,cur_te = iso_forest_mark_outliers(cur_tr,cur_te)
            cur_tr = np.array([mean_impute_ts(cur_tr[i]) for i in range(len(cur_tr))])
            cur_te = np.array([mean_impute_ts(cur_te[i]) for i in range(len(cur_te))])
            score("outlier",cur_tr,cur_te)

    return pd.DataFrame(records)


# ============================================================
# 11) RUN + EXPORT
# ============================================================

DATASETS = {
    "medical":   "medical_timeseries.npz",
    "education": "education_timeseries.npz",
    "usability": "usability_timeseries.npz",
}

all_results=[]
for ds,path in DATASETS.items():
    _,X_noisy,y = load_npz(path)
    for pipe_name,cfg in PIPELINES.items():
        df = eval_mv_first(X_noisy,y,cfg)
        df["dataset"]=ds
        all_results.append(df)

df_all = pd.concat(all_results,ignore_index=True)

acc_wide = df_all.groupby(["dataset","pipeline","step"])["accuracy"].mean().unstack("step")

out_xlsx="pipeline_stepwise_progressive_AO_MVfirst.xlsx"
acc_wide.to_excel(out_xlsx)

print("Saved:",out_xlsx)
display(acc_wide)


Saved: pipeline_stepwise_progressive_AO_MVfirst.xlsx


step                    mv  outlier   raw
dataset   pipeline                       
education A_baseline  0.52      NaN  0.48
          B_robust    0.52     0.50  0.48
          C_signal    0.52     0.50  0.48
          D_limit     0.52     0.50  0.48
          E_clinical  0.52      NaN  0.48
medical   A_baseline  0.30      NaN  0.42
          B_robust    0.30     0.36  0.42
          C_signal    0.30     0.36  0.42
          D_limit     0.30     0.36  0.42
          E_clinical  0.30      NaN  0.42
usability A_baseline  0.34      NaN  0.34
          B_robust    0.34     0.48  0.34
          C_signal    0.34     0.48  0.34
          D_limit     0.34     0.48  0.34
          E_clinical  0.34      NaN  0.34